In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

def advanced_preprocessing_phase1(df):
    df = df.copy()
    
    att_cols = [c for c in df.columns if 'Att_Subject' in c]
    
    for col in att_cols:
        df[col] = df[col].fillna(-1) # Fill NaN bằng -1 (chưa học)
        # Nếu khác -1, thì giá trị phải nằm trong [0, 16]
        mask = df[col] != -1
        df.loc[mask, col] = df.loc[mask, col].clip(0, 16)

    # Feature Engineering từ Chuyên cần:
    # Số lượng môn sinh viên thực sự đã học
    df['Actual_Subj_Count'] = (df[att_cols] > -1).sum(axis=1)
    
    # Tổng số buổi vắng mặt (Giả định mỗi môn tối đa 16 buổi)
    temp_att = df[att_cols].replace(-1, np.nan)
    df['Total_Absence'] = (16 - temp_att).sum(axis=1)
    df['Avg_Absence_Per_Subj'] = df['Total_Absence'] / (df['Actual_Subj_Count'] + 1)
    
    # Tỷ lệ đi học trung bình (Attendance Rate)
    df['Avg_Att_Rate'] = temp_att.mean(axis=1) / 16
    
    # Độ biến động chuyên cần (Std) - Sinh viên bất ổn thường có Std cao
    df['Att_Volatility'] = temp_att.std(axis=1).fillna(0)

    def clean_text_basic(series):
        return series.fillna('unknown').astype(str).str.lower().str.strip()

    cat_to_clean = ['Admission_Mode', 'Hometown', 'Gender', 'Club_Member']
    for col in cat_to_clean:
        df[col] = clean_text_basic(df[col])

    # Xử lý riêng English_Level cực sạch
    df['English_Level'] = clean_text_basic(df['English_Level'])
    df['English_Level'] = df['English_Level'].str.replace('.', '', regex=False) # Xóa dấu chấm (A1. -> a1)

    def map_english_robust(val):
        # Nhóm cao
        if any(kw in val for kw in ['ielts 6', 'ielts 6+', 'ielts 60', 'ielts 60+', 'ielts 6.0', 'ielts 6.0+', 'b2', 'c1', 'c2', 'high']):
            return 'ielts_high'
        # Nhóm thấp
        if any(kw in val for kw in ['a1', 'a2', 'b1', 'toeic', 'low']):
            return 'ielts_low'
        return 'unknown'

    df['English_Level'] = df['English_Level'].apply(map_english_robust)
    
    # ĐỊA LÝ & THÔNG TIN CÁ NHÂN
    # Tách Tỉnh/Thành từ địa chỉ hiện tại
    df['City_From_Address'] = df['Current_Address'].fillna('').apply(
        lambda x: x.split(',')[-1].strip().lower() if ',' in x else 'unknown'
    )
    
    # Đặc trưng "Học xa nhà": Hometown khác với Tỉnh đang ở hiện tại
    df['Is_Away_From_Home'] = (df['Hometown'] != df['City_From_Address']).astype(int)
    
    # Xử lý Age
    df['Age'] = df['Age'].clip(18, 50)
    df['Is_Mature_Student'] = (df['Age'] > 22).astype(int)

    # TÀI CHÍNH & HỌC THUẬT TƯƠNG TÁC
    df['Is_In_Debt'] = (df['Tuition_Debt'] > 0).astype(int)
    
    # Chỉ số áp lực tài chính: Nợ càng cao, điểm càng thấp -> Áp lực càng lớn
    df['Financial_Stress_Index'] = df['Tuition_Debt'] / (df['Training_Score_Mixed'] + 1)
    
    # Tương tác giữa việc học yếu (Count_F) và nợ học phí
    df['Academic_Financial_Stress'] = df['Count_F'] * (df['Tuition_Debt'] > 0).astype(int)

    return df

# Thực thi
train_p1 = advanced_preprocessing_phase1(train)
test_p1 = advanced_preprocessing_phase1(test)

print(f"--- Giai đoạn 1 hoàn tất ---")
print(f"Số lượng cột sau xử lý: {train_p1.shape[1]}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/competitions/is54a-tri-tu-nhan-to-bav-itde-2026/train.csv'

In [ ]:
import re

def clean_vntitle(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Xử lý các cụm Tiếng Anh phổ biến bằng cách chuẩn hóa dấu câu
    # Ví dụ: "warning: low attendance" -> "warning low attendance"
    text = re.sub(r'[:;,.!?]', ' ', text)
    # Từ điển sửa lỗi viết tắt/teencode phổ biến
    corrections = {
        r'\bko\b': 'không',
        r'\bk\b': 'không',
        r'\bhông\b': 'không',
        r'\bdc\b': 'được',
        r'\bđc\b': 'được',
        r'\be\b': 'em',
        r'\bj\b': 'gì',
        r'\bqa\b': 'quá',
        r'\bmk\b': 'mình',
        r'\bviec\b': 'việc',
        r'\bwarning\b': 'cảnh_báo',
        r'\battendance\b': 'chuyên_cần',
        r'\blow\b': 'thấp',
        r'\bthi lại\b': 'thi_lại',
        r'\bexcellent\b': 'tốt',
        r'\bbít\b': 'biết',
        r'\blun\b': 'luôn',
        r'\bm\b': 'mình',
        r'\bhông\b': 'không',
        r'\bhoc\b': 'học',
        r'\bnhung\b': 'nhưng',
        r'\bđồngg\b': 'đồng',
        r'\bcủaa\b': 'của',
        r'\bkhiếnn\b': 'khiến',
        r'\bhànhh\b': 'hành',
        r'\bnhg\b': 'nhưng',
        r'\bhiệnn\b': 'hiện',
        r'\blấyy\b': 'lấy',
        r'\btheoo\b': 'theo',
        r'\bchỉi\b': 'chỉ',
        r'\bviênn\b': 'viên',
        r'\bgiáá\b': 'giá',
        r'\bnhiệmm\b': 'nhiệm',
        r'\bnàoo\b': 'nào'
        
    }
    
    for pattern, replacement in corrections.items():
        text = re.sub(pattern, replacement, text)
        
    # Loại bỏ ký tự đặc biệt, chỉ giữ lại chữ và số
    text = re.sub(r'[^\w\s]', ' ', text)
    # Loại bỏ khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Áp dụng làm sạch
train_p1['Advisor_Notes_Clean'] = train_p1['Advisor_Notes'].apply(clean_vntitle)
train_p1['Personal_Essay_Clean'] = train_p1['Personal_Essay'].apply(clean_vntitle)

test_p1['Advisor_Notes_Clean'] = test_p1['Advisor_Notes'].apply(clean_vntitle)
test_p1['Personal_Essay_Clean'] = test_p1['Personal_Essay'].apply(clean_vntitle)

In [ ]:
def extract_advanced_text_stats(df):
    df = df.copy()
    
    # 1. Định nghĩa các nhóm từ khóa đặc thù (Bổ sung từ ví dụ của bạn)
    neg_advisor_words = [
        'không coi trọng', 'tự làm khó', 'mở rộng khoảng cách', 'không đến lớp', 'không chăm chỉ', 'làm lại từ đầu', 'lười biếng', 'cấm thi',
        'đánh cược', 'mạo hiểm', 'sa sút', 'cảnh báo', 'đình chỉ', 'yếu', 'kém', 'thấp', 'bỏ tiết', 'không hiệu quả', 'hậu quả nghiêm trọng',
        'lãng phí thời gian', 'học tập kém', 'nghiêm khắc', 'bỏ tiết kéo dài', 'căn bệnh', 'ý thức kém', 'đáng lo ngại', 'đánh cược', 'mạo hiểm',
        'không đến lớp', 'biến pháp mạnh mẽ hơn', 'vắng học', 'thường xuyên vắng học', 'tiếp tục bỏ tiết', 'cảnh_báo', 'thiếu ý thức', 'nghỉ học',
        'không có ý thức', 'chẳng còn ý nghĩa gì'
    ]
    
    neg_general_words = [
        'nghỉ', 'vắng', 'muộn', 'trốn', 'nợ', 'trượt', 'chán', 'nản', 'áp lực', 'bế tắc'
    ]
    
    pos_general_words = [
        'tốt', 'chăm', 'giỏi', 'nỗ lực', 'cố gắng', 'tiến bộ', 'kỷ luật', 'chuyên cần', 'xuất sắc', 'tích cực', 'tự giác', 'thích', 'tấm gương',
        'quý giá', 'tương lai', 'thú vị', 'kỷ niệm', 'ấm áp', 'phát triển', 'đem lại', 'đáng giá', 'kiên cường', 'đầu tư', 'chúc mừng', 'trái ngọt',
        'đánh giá cao', 'nể phục' , 'đáng khen ngợi', 'đáng khen', 'vui mừng', 'đáng trân trọng', 'cẩn thận', 'đi học đúng giờ', 'phát huy', 'hướng tới thành công',
        'đẩy đủ', 'tiếp tục cố gắng', 'phấn chấn', 'ước mơ', 'không từ bỏ'
    ]

    # Hàm đếm số từ xuất hiện
    def count_keywords(text, keywords):
        if not isinstance(text, str) or text == "": return 0
        return sum(1 for word in keywords if word in text.lower())

    # 2. Tính toán điểm số cho Advisor (Trọng số cao)
    df['Note_Neg_Score'] = df['Advisor_Notes_Clean'].apply(lambda x: count_keywords(x, neg_advisor_words + neg_general_words))
    df['Note_Pos_Score'] = df['Advisor_Notes_Clean'].apply(lambda x: count_keywords(x, pos_general_words))
    
    # 3. Tính toán điểm số cho Essay (Trọng số thấp hơn)
    df['Essay_Neg_Score'] = df['Personal_Essay_Clean'].apply(lambda x: count_keywords(x, neg_general_words))
    df['Essay_Pos_Score'] = df['Personal_Essay_Clean'].apply(lambda x: count_keywords(x, pos_general_words))

    # --- 4. TẠO CÁC ĐẶC TRƯNG THÔNG MINH ---

    # Chỉ số niềm tin của Giáo viên (Advisor Confidence)
    # Công thức: $$Score = (Note_{Pos} \times 3) - (Note_{Neg} \times 5)$$
    # (Trừ nặng hơn nếu có từ tiêu cực từ giáo viên)
    df['Advisor_Trust_Score'] = (df['Note_Pos_Score'] * 3) - (df['Note_Neg_Score'] * 5)

    # Flag mâu thuẫn: Sinh viên "nói hay" nhưng thực tế "học dở" (theo lời giáo viên)
    # Đây là đặc trưng cực mạnh cho nhóm Dropout
    df['Conflict_Flag'] = ((df['Essay_Pos_Score'] > 2) & (df['Note_Neg_Score'] > 0)).astype(int)

    # Đặc trưng "Báo động đỏ" từ giáo viên
    df['Advisor_Red_Flag'] = df['Advisor_Notes_Clean'].apply(
        lambda x: 1 if any(word in str(x).lower() for word in neg_advisor_words) else 0
    )

    # 5. Thống kê cơ bản
    df['Notes_Len'] = df['Advisor_Notes_Clean'].str.len()
    df['Essay_Len'] = df['Personal_Essay_Clean'].str.len()
    
    return df

# Áp dụng
train_p3 = extract_advanced_text_stats(train_p1)
test_p3 = extract_advanced_text_stats(test_p1)

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# 1. Thiết lập danh sách các đặc trưng
# Loại bỏ các cột không dùng để training
drop_cols = ['Student_ID', 'Academic_Status', 'Advisor_Notes', 'Personal_Essay', 'Current_Address']
features = [c for c in train_p3.columns if c not in drop_cols]

cat_features = ['Gender', 'Hometown', 'Admission_Mode', 'English_Level', 'Club_Member', 'City_From_Address']
text_features = ['Advisor_Notes_Clean', 'Personal_Essay_Clean']

X = train_p3[features]
y = train_p3['Academic_Status']
X_test = test_p3[features]

# 2. Cấu hình K-Fold
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Mảng lưu trữ dự đoán
oof_preds = np.zeros((len(X), 3)) # Lưu xác suất dự đoán trên tập Train (Out-of-fold)
test_preds = np.zeros((len(X_test), 3)) # Lưu xác suất dự đoán trên tập Test
fold_scores = []

print(f"--- Bắt đầu huấn luyện {n_splits}-Fold Cross Validation ---\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # Khởi tạo mô hình cho mỗi Fold
    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.02,
        depth=6,
        loss_function='MultiClass',
        auto_class_weights='Balanced',
        eval_metric='TotalF1',
        random_seed=42 + fold, # Thay đổi seed để tăng sự đa dạng
        # Cấu hình xử lý text nội bộ
        tokenizers=[
            {'tokenizer_id': 'Space', 'separator_type': 'ByDelimiter', 'delimiter': ' '}
        ],
        dictionaries=[
            {'dictionary_id': 'Word', 'gram_order': '1'},
            {'dictionary_id': 'BiGram', 'gram_order': '2'}
        ],
        feature_calcers=[
            'BoW:top_tokens_count=2000', # Tăng số lượng từ vựng quan trọng
            'NaiveBayes' # Thêm NaiveBayes để mô hình học xác suất từ/nhãn tốt hơn
        ],
        verbose=0 # Để verbose=0 để tránh trôi màn hình, ta sẽ in kết quả cuối mỗi fold
    )
    
    # Huấn luyện mô hình
    model.fit(
        X_train_fold, y_train_fold,
        cat_features=cat_features,
        text_features=text_features,
        eval_set=(X_val_fold, y_val_fold),
        early_stopping_rounds=100
    )
    
    # Lưu dự đoán cho tập Validation của Fold này
    val_probs = model.predict_proba(X_val_fold)
    oof_preds[val_idx] = val_probs
    
    # Cộng dồn dự đoán cho tập Test (sau đó chia trung bình)
    test_probs = model.predict_proba(X_test)
    test_preds += test_probs / n_splits
    
    # Đánh giá Fold hiện tại
    fold_labels = np.argmax(val_probs, axis=1)
    score = f1_score(y_val_fold, fold_labels, average='macro')
    fold_scores.append(score)
    
    print(f"Fold {fold+1}: Macro F1 = {score:.4f} | Best Iteration = {model.get_best_iteration()}")

# 3. Kết quả tổng quát
mean_score = np.mean(fold_scores)
std_score = np.std(fold_scores)
print(f"\n---> Trung bình CV Macro F1: {mean_score:.4f} (+/- {std_score:.4f})")

# In báo cáo chi tiết trên toàn bộ tập Train (OOF)
print("\nBáo cáo chi tiết trên tập OOF:")
print(classification_report(y, np.argmax(oof_preds, axis=1)))

--- Bắt đầu huấn luyện 5-Fold Cross Validation ---

Fold 1: Macro F1 = 0.8661 | Best Iteration = 306
Fold 2: Macro F1 = 0.8555 | Best Iteration = 553
Fold 3: Macro F1 = 0.8473 | Best Iteration = 512
Fold 4: Macro F1 = 0.8333 | Best Iteration = 416
Fold 5: Macro F1 = 0.8557 | Best Iteration = 355

---> Trung bình CV Macro F1: 0.8516 (+/- 0.0109)

Báo cáo chi tiết trên tập OOF:
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      3858
           1       0.84      0.77      0.80      1340
           2       0.70      0.85      0.77       802

    accuracy                           0.91      6000
   macro avg       0.84      0.87      0.85      6000
weighted avg       0.92      0.91      0.91      6000



In [ ]:
from sklearn.metrics import f1_score

def find_best_weights(y_true, y_probs):
    best_f1 = 0
    best_w1, best_w2 = 1.0, 1.0
    
    # Quét trọng số cho lớp 1 (Warning) và lớp 2 (Dropout)
    # Thường trọng số sẽ dao động quanh mức 1.0 đến 1.5
    for w1 in np.arange(1.0, 1.5, 0.05):
        for w2 in np.arange(1.0, 1.6, 0.05):
            preds = []
            for p in y_probs:
                # Logic Weighting Probabilities
                weighted_p = [p[0], p[1] * w1, p[2] * w2]
                preds.append(np.argmax(weighted_p))
            
            score = f1_score(y_true, preds, average='macro')
            if score > best_f1:
                best_f1 = score
                best_w1, best_w2 = w1, w2
                
    return best_w1, best_w2, best_f1

# 1. Tìm bộ trọng số tối ưu trên OOF
w1_opt, w2_opt, best_cv_f1 = find_best_weights(y, oof_preds)

print(f"--- KẾT QUẢ TỐI ƯU TRỌNG SỐ ---")
print(f"Trọng số W1 (Warning): {w1_opt:.2f}")
print(f"Trọng số W2 (Dropout): {w2_opt:.2f}")
print(f"Macro F1 tối ưu trên CV: {best_cv_f1:.4f}")

# 2. Áp dụng logic Weighting Argmax lên tập Test
final_labels = []

for p in test_preds:
    # Nhân trọng số đã tối ưu vào xác suất các lớp
    weighted_p = [p[0], p[1] * w1_opt, p[2] * w2_opt]
    
    # Lấy nhãn có xác suất (sau trọng số) lớn nhất
    final_labels.append(np.argmax(weighted_p))

# 3. Tạo file Submission
submission = pd.DataFrame({
    'Student_ID': test['Student_ID'],
    'Academic_Status': final_labels
})

submission.to_csv('submission_kfold_weighted.csv', index=False)
print("\nĐã lưu file: submission_kfold_weighted.csv")

# Kiểm tra phân bổ nhãn để đảm bảo tính thực tế
print("\nPhân bổ dự đoán mới (Weighting Argmax):")
print(submission['Academic_Status'].value_counts())
print(f"Tỷ lệ nhãn 2: {sum(np.array(final_labels)==2)/len(final_labels)*100:.2f}%")

--- KẾT QUẢ TỐI ƯU TRỌNG SỐ ---
Trọng số W1 (Warning): 1.25
Trọng số W2 (Dropout): 1.00
Macro F1 tối ưu trên CV: 0.8530

Đã lưu file: submission_kfold_weighted.csv

Phân bổ dự đoán mới (Weighting Argmax):
Academic_Status
0    2502
1     941
2     557
Name: count, dtype: int64
Tỷ lệ nhãn 2: 13.93%


In [ ]:
import pickle
with open('model_v1.pkl', 'wb') as f:
    pickle.dump(model, f)

NameError: name 'model' is not defined